In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import plotly.express as px

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import altair as alt

In [2]:
df = pd.read_csv("Data.csv")
df1= pd.read_csv("Series-Metadata.csv")

In [3]:
df

,Series Name,Country Name,2000,2001,2002,2003,2004,2005,2006,2007,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,"Adolescent fertility rate (births per 1,000 wo...",Afghanistan,154.31,152.471,150.161,143.75,136.93,130.838,121.635,117.754,...,84.069,81.043,78.13,75.3,73.021,70.967,68.877,66.599,65.339,64.068
1,"Adolescent fertility rate (births per 1,000 wo...",Albania,14.743,8.811,13.511,15.889,17.662,18.216,16.517,15.882,...,21.767,20.656,18.518,16.778,15.456,14.949,14.004,13.002,12.827,12.789
2,"Adolescent fertility rate (births per 1,000 wo...",Algeria,8.658,8.065,7.74,7.727,7.789,7.87,8.403,8.795,...,10.196,10.294,10.5,9.978,9.495,10.328,9.901,9.343,8.977,8.698
3,"Adolescent fertility rate (births per 1,000 wo...",American Samoa,48.376,46.153,44.466,42.685,41.905,38.191,35.852,33.607,...,43.059,42.174,39.45,37.274,35.427,34.278,34.149,34.214,34.104,34.181
4,"Adolescent fertility rate (births per 1,000 wo...",Andorra,9.075,9.84,9.384,8.335,8.712,9.064,9.383,9.034,...,4.947,4.936,4.576,4.326,3.832,3.526,3.119,3.5,3.527,3.48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23865,Young people (ages 15-24) newly infected with HIV,Virgin Islands (U.S.),,,,,,,,,...,,,,,,,,,,
23866,Young people (ages 15-24) newly infected with HIV,West Bank and Gaza,,,,,,,,,...,,,,,,,,,,
23867,Young people (ages 15-24) newly infected with HIV,"Yemen, Rep.",200,200,200,200,200,200,200,200,...,500,500,500,500,500,500,500,500,500,
23868,Young people (ages 15-24) newly infected with HIV,Zambia,27000,27000,27000,27000,26000,26000,25000,25000,...,25000,23000,22000,23000,23000,20000,16000,14000,12000,


begin cleaning

In [4]:
cols_to_convert = [col for col in df.columns if col not in ["Series Name", "Country Name"]]
df[cols_to_convert] = df[cols_to_convert].apply(pd.to_numeric, errors="coerce").astype(float)
print(df.dtypes)

Series Name      object
Country Name     object
2000            float64
2001            float64
2002            float64
2003            float64
2004            float64
2005            float64
2006            float64
2007            float64
2008            float64
2009            float64
2010            float64
2011            float64
2012            float64
2013            float64
2014            float64
2015            float64
2016            float64
2017            float64
2018            float64
2019            float64
2020            float64
2021            float64
2022            float64
2023            float64
dtype: object


In [5]:
print("df missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("df1 missing values:")
print(df1.isnull().sum()[df1.isnull().sum() > 0])

df missing values:
2000     5137
2001     5162
2002     5076
2003     4757
2004     4647
2005     4664
2006     4638
2007     4626
2008     4576
2009     4554
2010     4487
2011     4497
2012     4512
2013     4454
2014     4446
2015     4476
2016     4456
2017     4498
2018     4496
2019     4629
2020     4889
2021     5634
2022     7609
2023    14959
dtype: int64
df1 missing values:
Unit of measure               11
Aggregation method             6
Development relevance         11
Limitations and exceptions    37
dtype: int64


connect to metadata

In [6]:
df1 = df1.iloc[:, [0]]

In [7]:
percent_indicators = [
    "Immunization, DPT (% of children ages 12-23 months)",
    "Immunization, HepB3 (% of one-year-old children)",
    "Immunization, measles (% of children ages 12-23 months)",
    "People using at least basic drinking water services (% of population)",
    "People using at least basic drinking water services, rural (% of rural population)",
    "People using at least basic drinking water services, urban (% of urban population)",
    "People using at least basic sanitation services (% of population)",
    "People using at least basic sanitation services, rural (% of rural population)",
    "People using at least basic sanitation services, urban (% of urban population)",
    "People using safely managed drinking water services (% of population)",
    "People using safely managed drinking water services, rural (% of rural population)",
    "People using safely managed drinking water services, urban (% of urban population)",
    "People using safely managed sanitation services (% of population)",
    "People using safely managed sanitation services, rural (% of rural population)",
    "People using safely managed sanitation services, urban (% of urban population)",
    "Tuberculosis case detection rate (%, all forms)",
    "Tuberculosis treatment success rate (% of new cases)",
    "Current health expenditure (% of GDP)",
    "Domestic general government health expenditure (% of current health expenditure)",
    "Domestic general government health expenditure (% of GDP)",
    "Domestic general government health expenditure (% of general government expenditure)",
    "Domestic private health expenditure (% of current health expenditure)",
    "External health expenditure (% of current health expenditure)",
    "Out-of-pocket expenditure (% of current health expenditure)",
    "Mortality from CVD, cancer, diabetes or CRD between exact ages 30 and 70 (%)",
    "Mortality from CVD, cancer, diabetes or CRD between exact ages 30 and 70, female (%)",
    "Mortality from CVD, cancer, diabetes or CRD between exact ages 30 and 70, male (%)",
    "Survival to age 65, female (% of cohort)",
    "Survival to age 65, male (% of cohort)",
    "Age dependency ratio (% of working-age population)",
    "Age dependency ratio, old (% of working-age population)",
    "Age dependency ratio, young (% of working-age population)",
    "Population growth (annual %)",
    "Births attended by skilled health staff (% of total)",
    "Lifetime risk of maternal death (%)",
    "Antiretroviral therapy coverage (% of people living with HIV)",
    "Prevalence of HIV, female (% ages 15-24)",
    "Prevalence of HIV, male (% ages 15-24)",
    "Prevalence of HIV, total (% of population ages 15-49)",
    "Risk of catastrophic expenditure for surgical care (% of people at risk)",
    "Risk of impoverishing expenditure for surgical care (% of people at risk)",
    "Women's share of population ages 15+ living with HIV (%)"
]

df1["Data Type"] = df1["Indicator Name"].apply(lambda x: "%" if x in percent_indicators else "Number")

cleaning time series data

In [8]:
year_cols = [str(y) for y in range(2000, 2024)]

df_cleaned = df.dropna(subset=year_cols, how="all")
df_cleaned = df_cleaned[df_cleaned[year_cols].notna().sum(axis=1) > 1]

print(f"Original rows: {len(df)}")
print(f"After removing all-null and single-value rows: {len(df_cleaned)}")

Original rows: 23870
After removing all-null and single-value rows: 19876


In [9]:
df_cleaned.isna().sum()

Series Name         0
Country Name        0
2000             1148
2001             1170
2002             1084
2003              764
2004              655
2005              674
2006              650
2007              645
2008              582
2009              562
2010              494
2011              504
2012              519
2013              465
2014              455
2015              482
2016              464
2017              504
2018              506
2019              636
2020              895
2021             1640
2022             3620
2023            10965
dtype: int64

In [10]:
df_cleaned = df_cleaned.drop(columns=["2023"])

In [11]:
df_long = df_cleaned.melt(
    id_vars=["Series Name", "Country Name"],
    var_name="Year",
    value_name="Value"
)

In [12]:
df_long

,Series Name,Country Name,Year,Value
0,"Adolescent fertility rate (births per 1,000 wo...",Afghanistan,2000,154.310
1,"Adolescent fertility rate (births per 1,000 wo...",Albania,2000,14.743
2,"Adolescent fertility rate (births per 1,000 wo...",Algeria,2000,8.658
3,"Adolescent fertility rate (births per 1,000 wo...",American Samoa,2000,48.376
4,"Adolescent fertility rate (births per 1,000 wo...",Andorra,2000,9.075
...,...,...,...,...
457143,Young people (ages 15-24) newly infected with HIV,Uruguay,2022,200.000
457144,Young people (ages 15-24) newly infected with HIV,Viet Nam,2022,2100.000
457145,Young people (ages 15-24) newly infected with HIV,"Yemen, Rep.",2022,500.000
457146,Young people (ages 15-24) newly infected with HIV,Zambia,2022,12000.000


In [13]:
df_long["Year"] = pd.to_numeric(df_long["Year"])

In [14]:
df_long["Series Name"].unique()

array(['Adolescent fertility rate (births per 1,000 women ages 15-19)',
       'Adults (ages 15+) and children (ages 0-14) newly infected with HIV',
       'Adults (ages 15-49) newly infected with HIV',
       'Age dependency ratio (% of working-age population)',
       'Age dependency ratio, old (% of working-age population)',
       'Age dependency ratio, young (% of working-age population)',
       'Antiretroviral therapy coverage (% of people living with HIV)',
       'Birth rate, crude (per 1,000 people)',
       'Births attended by skilled health staff (% of total)',
       'Children (0-14) living with HIV',
       'Children (ages 0-14) newly infected with HIV',
       'Current health expenditure (% of GDP)',
       'Current health expenditure per capita (current US$)',
       'Current health expenditure per capita, PPP (current international $)',
       'Death rate, crude (per 1,000 people)',
       'Domestic general government health expenditure (% of current health expenditure

In [15]:
df_exp = df_long[(df_long["Series Name"] == "Current health expenditure (% of GDP)")]

In [16]:
df_exp

,Series Name,Country Name,Year,Value
1859,Current health expenditure (% of GDP),Afghanistan,2000,NaN
1860,Current health expenditure (% of GDP),Albania,2000,5.944198
1861,Current health expenditure (% of GDP),Algeria,2000,3.214854
1862,Current health expenditure (% of GDP),Andorra,2000,5.952764
1863,Current health expenditure (% of GDP),Angola,2000,1.908599
...,...,...,...,...
439318,Current health expenditure (% of GDP),Viet Nam,2022,4.588916
439319,Current health expenditure (% of GDP),West Bank and Gaza,2022,9.733360
439320,Current health expenditure (% of GDP),"Yemen, Rep.",2022,6.186667
439321,Current health expenditure (% of GDP),Zambia,2022,5.255923


In [17]:
df_exp = df_exp[df_exp["Year"] == 2022]

In [18]:
df_exp

,Series Name,Country Name,Year,Value
439131,Current health expenditure (% of GDP),Afghanistan,2022,23.088169
439132,Current health expenditure (% of GDP),Albania,2022,6.193681
439133,Current health expenditure (% of GDP),Algeria,2022,3.623043
439134,Current health expenditure (% of GDP),Andorra,2022,7.536788
439135,Current health expenditure (% of GDP),Angola,2022,2.927376
...,...,...,...,...
439318,Current health expenditure (% of GDP),Viet Nam,2022,4.588916
439319,Current health expenditure (% of GDP),West Bank and Gaza,2022,9.733360
439320,Current health expenditure (% of GDP),"Yemen, Rep.",2022,6.186667
439321,Current health expenditure (% of GDP),Zambia,2022,5.255923


In [19]:
fig = px.choropleth(
    df_exp,
    locations="Country Name",
    locationmode="country names",
    color="Value",
    hover_name="Country Name",
    color_continuous_scale="Blues",
    title="Health Expenditure per Capita (2022)"
)
fig.write_html("choropleth_map.html")
fig.show()

C:\Users\kamat\AppData\Local\Temp\ipykernel_13916\2082001384.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


(D3) VISUALIZATION 2: scatterplot expenditure vs life expectancy

In [20]:
df_filtered = df_long[
    df_long["Series Name"].isin([
        "Current health expenditure (% of GDP)",
        "Life expectancy at birth, total (years)"
    ])
]

In [21]:
df_wide = df_filtered.pivot_table(
    index=["Country Name", "Year"],
    columns="Series Name",
    values="Value"
).reset_index()

In [22]:
df_wide = df_wide.dropna()

# Rename columns to simple names (VERY important)
df_wide = df_wide.rename(columns={
    "Country Name": "Country",
    "Current health expenditure (% of GDP)": "Spending",
    "Life expectancy at birth, total (years)": "LifeExpectancy"
})

df_wide = df_wide[df_wide["Spending"] > 0]

df_wide.to_csv("D3_data.csv", index=False)

In [23]:
d3_data = pd.read_csv("C:/Users/carli/Documents/fourth year/DS4200/project/D3_data.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/carli/Documents/fourth year/DS4200/project/D3_data.csv'

In [ ]:
d3_data.head()

In [ ]:
d3_data["Country"].dtype

(D3) VISUALIZATION 3: line plot aggregated total spending vs time

In [ ]:
df_exp_dollars= df_long[(df_long["Series Name"] == "Current health expenditure per capita (current US$)")]

In [ ]:
df_exp_dollars.dropna(subset=["Value", "Year"])

In [ ]:
year_sum = df_exp_dollars.groupby("Year")["Value"].sum().reset_index()

In [ ]:
year_sum

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(year_sum["Year"], year_sum["Value"], marker='o', linewidth=2)

plt.xlabel("Year")
plt.ylabel("Total Health Expenditure")
plt.title("Total Health Expenditure Over Time (2000–2022)")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()